* Here , I am using market_pipeline_env named virtual environment for my project .

* I have installed pandas numpy matplotlib yfinance (simple libraries) as this project is purely learning based. I have not used any sort of fancy libraries. We will be keeping it simple for now

* after that i have use pip freeze>requirements.txt

IMPORTANT :

Jupyter notebooks do not share memory with each other. So , if u want to use same datasets again and again, you better make a python script file for that.

Notebooks should not pass data to each other . Instead, the data pipeline should be implemented through reusable Python modules(functions).

Why?

Jupyter notebooks run in separate kernels, so variables created in one notebook do not persist in another.

Because of this, relying on notebooks to pass datasets between stages leads to fragile nad non-reproductive workflows.

In a financial research pipeline, data acquisition should be separated
from analysis. Data fetching scripts are responsible for collecting
and storing raw market data, while research notebooks focus only on
analysis such as cleaning, return computation, and volatility modeling.

This separation ensures reproducibility, scalability, and cleaner
research workflows.

A robust financial data pipeline separates data acquisition
(data_fetcher) from data access (data_loader). Fetchers
collect and store raw data from external sources, while
loaders provide a standardized interface to access the
stored data for research and analysis.

1. <strong> FETCHER SCRIPT </strong> 

Why use Parquet instead of CSV? 

-> Both store tabular data , but they are very different internally.

CSV(text format)

A CSV looks like:
Date,Open,High,Low,Close,Volume
2024-01-01,100,105,98,102,123456
2024-01-02,102,107,100,106,130000

Characteristics:
* plain text
* human readable
* larger file size
* slow to load
* no schema/datatype info

Example problems:
* numbers may load as strings
* timestamps sometimes misparsed
* large datasets become slow


Parquet(columnar binary format)

Parquet is a compressed binary columnar storage format used in:

* quantitative finance
* data engineering
* machine learning pipelines
* big data systems(Spark,Hadoop)

Instead of storing rows, it stores columns.

Conceptually:
Column: Date
Column: Open
Column: High
Column: Low
Column: Close
Column: Volume

Performance Comparison

| Format  | File Size | Load Speed |
| ------- | --------- | ---------- |
| CSV     | 100 MB    | slow       |
| Parquet | 20–30 MB  | very fast  |


* Conclusion 

for pipeline :
raw data storage → parquet

for export or sharing:
analysis output → csv


In fetcher we have made a Class so that we can use it it in some other project as well and this is one of the important use case of this project.


NOTE: 
The next improvements that make this quant-grade infrastructure are: (Do it a try)

1️.   Retry logic for API failures

2️.   Automatic incremental updates

3️.   parallel downloading for hundreds of tickers -> use multithreading

4️.   data validation checks

2. LOADER SCRIPT


In data pipelines, data acquisition (fetcher) and data
access (loader) should be separated. Fetchers collect
data from external sources, while loaders provide a
standard interface to retrieve stored datasets for
analysis.

In [2]:
# Instead of doing :

import pandas as pd
# df = pd.read_parquet("../data/raw/parquet/ADANIPORTS.NS.parquet")

# display(df.head())

# loading the csv file
# file_path = D:\Quant_Daily_Learning\projects\Market Data Pipeline\data\raw\csv\ADANIPORTS_NS_1d.csv
df = pd.read_csv("../data/raw/csv/ADANIPORTS_NS_1d.csv")

display(df.head())


,Date,Open,High,Low,Close,Volume
0,2026-01-01,1472.800049,1490.900024,1470.400024,1481.099976,981252
1,2026-01-02,1480.099976,1495.000000,1478.300049,1489.500000,1251176
2,2026-01-05,1488.000000,1496.800049,1479.099976,1493.000000,790039
3,2026-01-06,1490.000000,1493.000000,1470.699951,1473.199951,1675609
4,2026-01-07,1473.199951,1480.400024,1459.000000,1465.300049,2124534


In [3]:
# use 

import sys
import os

# Add project root directory to sys.path
project_root = os.path.abspath(os.path.abspath('..'))
sys.path.append(project_root)

# uding the data loader class to load the data instead of directly using pandas. This way we can have a more modular and reusable code.
from scripts.data_loader import DataLoader
loader = DataLoader(file_format="parquet")

df = loader.load_ticker_data("ADANIPORTS.NS","1d")

display(df.head())

Price,Open,High,Low,Close,Volume
Date,,,,,
2025-03-11,1131.793674,1140.352011,1118.956106,1129.504761,2399428
2025-03-12,1133.485427,1149.557259,1099.650041,1122.837280,2842848
2025-03-13,1129.504802,1133.933208,1109.651440,1113.781250,1358839
2025-03-17,1109.402643,1142.989239,1105.819976,1131.644409,1489537
2025-03-18,1131.644409,1131.644409,1131.644409,1131.644409,0


In [4]:
tickers = ["RELIANCE.NS","TCS.NS"]

df = loader.load_multiple_tickers(tickers, "1d")

print(df.keys())

display(df["RELIANCE.NS"].head())

dict_keys(['RELIANCE.NS', 'TCS.NS'])


Price,Open,High,Low,Close,Volume
Date,,,,,
2025-03-11,1235.067248,1244.529457,1224.858023,1242.338257,10089838
2025-03-12,1252.995611,1257.975720,1239.051303,1252.049438,11977555
2025-03-13,1255.037605,1259.121271,1238.901975,1242.935913,9505270
2025-03-17,1237.208674,1252.398009,1228.194627,1233.921753,16640917
2025-03-18,1233.921753,1233.921753,1233.921753,1233.921753,0


Right now your loader returns one dataframe per ticker or a stacked dataframe. But for quant analysis we often need aligned price matrices.

Example structure used in finance:


Date            RELIANCE.NS     TCS.NS        INFY.NS

2026-01-01        2500          3900           1500

2026-01-02        2520          3880           1512

2026-01-03        2515          3920           1508

This format is critical because:

returns

covariance

correlation

portfolio weights

all require aligned time series.

In [5]:
tickers = ["RELIANCE.NS","TCS.NS","INFY.NS"]

prices = loader.load_price_matrix(tickers, "1d")

print(prices.head())

            RELIANCE.NS  TCS.NS  INFY.NS
Date                                    
2025-03-11  1242.338257     NaN      NaN
2025-03-12  1252.049438     NaN      NaN
2025-03-13  1242.935913     NaN      NaN
2025-03-17  1233.921753     NaN      NaN
2025-03-18  1233.921753     NaN      NaN


In [6]:
returns = prices.pct_change()
print(returns.head())

            RELIANCE.NS  TCS.NS  INFY.NS
Date                                    
2025-03-11          NaN     NaN      NaN
2025-03-12     0.007817     NaN      NaN
2025-03-13    -0.007279     NaN      NaN
2025-03-17    -0.007252     NaN      NaN
2025-03-18     0.000000     NaN      NaN


In [7]:
cov = returns.cov()
print(cov.head())

             RELIANCE.NS    TCS.NS   INFY.NS
RELIANCE.NS     0.000158  0.000011 -0.000005
TCS.NS          0.000011  0.000337  0.000281
INFY.NS        -0.000005  0.000281  0.000372


In [8]:
corr = returns.corr()   
print(corr.head())

             RELIANCE.NS    TCS.NS   INFY.NS
RELIANCE.NS     1.000000  0.041195 -0.017588
TCS.NS          0.041195  1.000000  0.791992
INFY.NS        -0.017588  0.791992  1.000000


### Some of the modifications that i have done in the fetcher scrpts (with problem and the solution) : 


1. Imagine the situation :

You run:

Ticker : RELIANCE.NS
Start : 2024-01-01
End : 2024-03-01
Interval : 1d

-> Saved file : RELIANCE_NS_1d.parquet


Later you run :

Ticker : RELIANCE.NS
Start : 2024-01-01
End : 2024-06ss-01
Interval : 1d

Now, if i save directly , the file will contain duplicate rows.

This breaks:

* return calculation

* rolling statistics

* covariance estimation

* backtests

So, we won't overwrite the data blindly.

Instead:

check if file exist

load existing data

find last timestamps

download only new data

append

remove duplicates

save

2. Missing candles in Market Data

APIs sometimes return incomplete data.

2024-01-01    2450

2024-01-02    2450

2024-01-05    2450

Missing: 2024-01-03 and 2024-01-04

Why is it dangerous?

Returns = P_t / P_(t-1)

if jan 2 -> jan 5 

The return is 3.days return instead of 1-day return.

Your statistics become wrong

<strong>Solution : Data Validation </strong>

After downloading the data, our pipeline should validate the time series.

Three checks :

1. Missing timestamps

2. Duplicate timestamps

3. Price anomalies

Professional systems always run data quality checks:

Raw Data

   ↓

Validation Layer

   ↓

Clean Dataset

   ↓

Feature Engineering

   ↓

Models




Without this step, even hedge funds get garbage signals.



### DATA FOLDER:

API
 ↓

RAW DATA

(Open High Low Close Volume)

 ↓

CLEANED DATA

(validated OHLCV)

 ↓


PROCESSED DATA

(returns, volatility, indicators)


fetch_data.py

        ↓

data/raw

clean_data.py

        ↓

data/cleaned

feature_engineering.py

        ↓
        
data/processed

analysis notebooks


### Data Validation Checkpoints in the Market Data Pipeline

These checkpoints are added to ensure financial data integrity before computing returns, volatility, and other statistics.

1️. Datetime Index Validation

Problem

Market data must be ordered by time. If the index is not a proper datetime index:

*  returns become incorrect

* rolling statistics break

* time series models fail

Example bad case:

2024-01-03

2024-01-01

2024-01-02

Solution

Convert and enforce datetime index.

df.index = pd.to_datetime(df.index)


2. Timestamp Sorting

Problem

Time series must be in chronological order.

Unsorted timestamps produce incorrect return calculations.

Solution: 
df.sort_index(inplace=True)

3. Duplicate Timestamp Removal

Problem

APIs sometimes return duplicate timestamps:

2024-01-01

2024-01-02

2024-01-02

2024-01-03

This breaks:

return calculations

rolling statistics

Solution: 

df = df[~df.index.duplicated(keep="last")]


4. Missing Timestamp Detection

Problem

Sometimes APIs skip candles.

This causes problems when building multi-asset price matrices.

Solution:
Check missing timestamps using expected trading frequency.

expected_dates = pd.date_range(start=start, end=end, freq="B")
missing_dates = expected_dates.difference(df.index)

5. Extreme Return Detection

Problem

Sometimes API errors create unrealistic returns.

100 → 10000

This corrupts:

volatility

Sharpe ratio

risk metrics

Solution:

returns = df["Close"].pct_change()
extreme = returns.abs() > 0.5

If detected:

log warning
investigate data

Final Data Quality Checklist:


✔ datetime index validation

✔ timestamp sorting

✔ duplicate timestamp removal

✔ missing timestamp detection

✔ NaN row removal

✔ forward filling gaps

✔ price positivity check

✔ extreme return detection


Missing timestamp validation ensures that financial time series do not
contain unexpected gaps in trading data. The validation compares the
dataset timestamps with an expected trading calendar (currently business
days). If timestamps are missing, it may indicate incomplete API data or
data corruption.

However, legitimate market closures such as weekends and exchange
holidays may also appear as missing timestamps. For research pipelines
this is acceptable, but production systems typically use exchange
calendars to handle holidays correctly.


Yahoo Finance data must be handled carefully because raw price data may
contain distortions caused by corporate actions such as stock splits
and dividends. If unadjusted prices are used, return calculations may
produce false extreme values. To avoid this issue, the pipeline uses
auto_adjust=True during data download so that historical prices are
adjusted for corporate actions before computing financial features.
